# 7. 自研 A 股日频回测：事件、账户、撮合与评价

本 notebook 接入 `signal&conbination.ipynb` 已保存的信号，完成 **目标 → 订单 → 实际成交 → 现金与持仓 → 净值 → 对照报告**。它不重新训练模型，也不把未来标签平均值直接当成策略收益。

每个主要步骤均说明输入、处理、输出与检查方法。函数按“数据适配、账户、费用与规则、订单规划、撮合、事件循环、评价”组织，方便以后提取到 `src/backtest/`。

**当前默认结果等级是 `research_price_return`：使用原始价格的研究账本。** 项目尚缺完整历史 ST/停复牌及分红送转事件，因此默认不伪造这些信息，也不自动从复权因子推算分红。未建模复权变化单独审计。这里的收益不能解释为完整股息再投资总收益，日成交量限制也不等于真实收盘竞价可成交量。

已提供公司行动事件接口与严格数据检查；严格模式在关键输入不足时停止。先运行手工账本验收和 OLS 首季度试运行，完整六模型回测按需开启。

## 1. 依赖与可调整的配置

**输入**：项目路径、成功的信号运行、回测区间和规则假设。

首版固定日频、只做多、独立模型账户、无杠杆。信号日收盘后产生订单，下一交易日收盘参考成交；当天最终价格只在实际撮合时读取。

先定义底层函数和默认配置，再单独设置试运行开关。这里仅导入 Python 模块，不运行训练、信号重建或回测。请使用安装 pandas、numpy、pyarrow、matplotlib 的 `factor_research` 环境。

In [ ]:
import ast
import copy
import hashlib
import json
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd

BACKTEST_SCHEMA="ashare_daily_event_backtest_v1"


def default_backtest_config():
    return dict(engine_version=BACKTEST_SCHEMA,models=["ols"],start="2025-01-02",end="2025-03-31",
        initial_cash=1_000_000.0,mode="research",corporate_action_mode="price_only",
        actions_coverage_verified=False,commission_rate=0.0003,min_commission=5.0,
        slippage_bps=5.0,impact_eta=0.1,volume_participation=0.01,planning_adv_limit=0.01,
        annual_days=252,risk_free_annual=0.0,progress_every=20,zero_cost_test=False,
        portfolio=dict(policy="topk_buffer",top_k=50,max_discretionary_replacements=5,
                       target_stock_weight=0.95,single_name_weight_cap=0.03,min_cross_section=30))


def load_signal_library(notebook_path):
    """只导入本项目可信 notebook 的定义，不运行导出、预览或训练单元格。"""
    notebook_path=Path(notebook_path)
    document=json.loads(notebook_path.read_text(encoding="utf-8-sig"))
    namespace={"__name__":"signal_contract_library"}
    constants={"SIGNAL_SCHEMA","EXPECTED_MODELS","FORBIDDEN_DECISION_FIELDS"}
    for index,cell in enumerate(document["cells"]):
        if cell["cell_type"]!="code":continue
        tree=ast.parse("".join(cell["source"]))
        selected=[]
        for node in tree.body:
            if isinstance(node,(ast.Import,ast.ImportFrom,ast.FunctionDef,ast.ClassDef)):
                selected.append(node)
            elif isinstance(node,ast.Assign) and all(isinstance(t,ast.Name) and t.id in constants for t in node.targets):
                ast.literal_eval(node.value)
                selected.append(node)
        exec(compile(ast.Module(body=selected,type_ignores=[]),f"signal_definitions_{index}","exec"),namespace)
    required={"load_signal_data","build_target_weights","estimate_target_shares","stable_hash",
              "stream_sha256","save_json_atomic","read_json_file","resolve_latest","as_shanghai",
              "strategy_identity","rule_for_instrument","legal_order_quantity"}
    if not required.issubset(namespace):raise ValueError("信号 notebook 缺少约定接口。")
    return namespace


def load_backtest_source(project_dir, library, signal_run=None):
    root=Path(project_dir)/"data/processed/signals"
    run=Path(signal_run).resolve() if signal_run is not None else library["resolve_latest"](root)
    metadata=library["read_json_file"](run/"metadata.json")
    config=library["read_json_file"](run/"config.json")
    if metadata.get("schema")!="signal_dataset_v1" or metadata.get("status")!="complete":
        raise ValueError("信号运行未完成或接口版本不匹配。")
    if library["stable_hash"](config)!=metadata["config_hash"]:raise ValueError("信号配置被修改。")
    for relative,digest in metadata["files_sha256"].items():
        if library["stream_sha256"](run/relative)!=digest:raise ValueError(f"信号分区校验失败：{relative}")
    report=pd.read_csv(run/"reports/daily_coverage.csv",parse_dates=["signal_date"])
    manifest=dict(signal_run=str(run),signal_metadata_sha256=library["stream_sha256"](run/"metadata.json"),
        signal_config_sha256=library["stream_sha256"](run/"config.json"),
        signal_source_sha256=library["stream_sha256"](run/"source.json"),
        signal_notebook_sha256=library["stream_sha256"](Path(project_dir)/"signal&conbination.ipynb"))
    return dict(project_dir=Path(project_dir),run=run,metadata=metadata,config=config,report=report,
                models=metadata["models"],manifest=manifest,source_hash=library["stable_hash"](manifest))


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import contextlib
import io as text_io

PROJECT_DIR = Path(r"E:\document-prefer\python for finanace\multi-model-bactest")
SIGNAL_RUN_OVERRIDE = None  # 可填写某个固定的 signals/run_... 目录
OUTPUT_ROOT = PROJECT_DIR / "data/processed/backtest"

HISTORICAL_STATUS_PATH = None
CORPORATE_ACTIONS_PATH = None
HISTORICAL_QUANTITY_RULES_PATH = None

RUN_ACCEPTANCE_TESTS = True
RUN_PILOT = True
RUN_FULL_BENCHMARK = True

BT_CONFIG = default_backtest_config()
# 默认 OLS、2025年第一季度、100万元。它是固定接口试运行，不是按测试收益选模型。
# 完整六模型比较见后面 RUN_FULL_BENCHMARK 开关。
display(pd.Series({k:v for k,v in BT_CONFIG.items() if k!="portfolio"}, name="value").to_frame())
display(pd.Series(BT_CONFIG["portfolio"], name="value").to_frame())


### 这些参数分别控制什么

| 参数 | 默认值 | 含义 |
| --- | --- | --- |
| `initial_cash` | 1000000 | 每个模型单独拥有的起始资金，单位元 |
| `mode` | research | 使用明确标记的研究假设；strict 需要完整核验数据 |
| `corporate_action_mode` | price_only | 默认价格收益账本；event_table 接入实际事件 |
| `commission_rate` / `min_commission` | 0.0003 / 5元 | 券商佣金的实验假设，不是所有账户的统一事实 |
| `slippage_bps` | 5 | 额外执行损耗假设；1bp=0.0001 |
| `impact_eta` | 0.1 | 未校准的冲击敏感性参数，初版作研究情景使用 |
| `volume_participation` | 0.01 | 模拟成交最多使用该股当日日成交量的1%，属于日线容量代理 |
| `planning_adv_limit` | 0.01 | 规划买单最多为信号日已知20日平均成交额的1% |
| `portfolio` | Top-50 / 最多轮换5只 | 沿用信号模块；已有持仓不假定全部换成目标 |

实际费用分别保存为佣金、印花税、过户费和执行损耗，避免把滑点重复计入价格与费用。默认使用原始收盘价记账，将执行损耗作为额外现金成本单列。

回测配置、数据来源和结果等级一起保存。已经评价过的2025年用于本阶段开发回放；后续若据此调整策略，新版本仍需新的未见期验证。

## 2. 只导入信号接口，固定成功产物

**输入**：本项目可信的 `signal&conbination.ipynb` 和 signals/latest 指针。

**处理**：从已有 notebook 只导入函数、类、import 及约定常量，不执行其导出、预览或测试单元格。因此不会重做745万行信号。函数保持在独立命名空间 `signal_lib` 中。

固定具体成功运行后检查所有信号分区的内容哈希。信号 notebook 自身也纳入来源记录；模型、日期与来源版本不会随着 latest 改变而在本次运行中途切换。

**输出**：`signal_lib`、`bt_source` 和信号覆盖报告。此处只读取分数与资格，不把 label、evaluation_label 等未来监督字段带入交易引擎。

In [ ]:
signal_lib = load_signal_library(PROJECT_DIR / "signal&conbination.ipynb")
bt_source = load_backtest_source(PROJECT_DIR, signal_lib, SIGNAL_RUN_OVERRIDE)
print("信号版本：", bt_source["run"])
print("来源等级：", bt_source["metadata"]["result_level"])
display(bt_source["report"].groupby("model").agg(
    start=("signal_date","min"), end=("signal_date","max"), rows=("rows","sum")))


## 3. 行情、历史规则与公司行动适配

**输入**：未复权日线、复权因子、交易日历、上市日期；可选的历史状态、公司行动和数量规则表。

**处理**：读取试运行区间及前30个交易日预热，成交量从手转股、成交额从千元转元。20日平均成交额和波动只用每个日期之前及当日已经发生的数据；规划 t 日订单只取 t 日这一行，执行 t+1 时才读 t+1 行。

复权因子只用于历史波动估计和未建模公司行动审计，不自动改变原始股数或生成现金分红。当前股票名称、当前ST状态和行业也不冒充完整历史状态。

**可选表接口**：

- 历史状态：`signal_date, instrument, available_at, historical_status_verified, known_can_buy, known_can_sell, upper_limit, lower_limit`。布尔列需明确；状态不得在对应成交参考时点之后才可用。
- 数量规则：`board, effective_from, effective_to, buy_min, buy_step, sell_min, sell_step, verified`。board 为 main/gem/star。
- 公司行动：通用字段 `event_id, event_type, event_date, instrument`；支持 split、dividend_record、cash_settlement，细节见后文。

**输出**：`bt_market`。strict 模式要求严格资格信号、完整历史状态、数量规则和确认覆盖的公司行动事件；缺失时明确停止，不静默放宽。

股票范围继承上游信号，并以现有 stock_basic 文件匹配上市日期；这份当前快照不能证明已覆盖历史退市股票。若下载范围遗漏退市证券，仍存在幸存者偏差，需要从原始股票池开始补齐，回测引擎无法自动消除。

In [ ]:
def load_backtest_market(source,library,start,end,status_path=None,actions_path=None,rules_path=None,warmup=30):
    root=source["project_dir"]/"data/raw"
    calendar_files=sorted((root/"basic").glob("trade_dates_*.csv"))
    calendar=pd.DatetimeIndex(pd.concat([pd.to_datetime(pd.read_csv(p,dtype=str,encoding="utf-8-sig")["trade_date"],format="%Y%m%d")
                                        for p in calendar_files]).drop_duplicates().sort_values())
    start,end=pd.Timestamp(start),pd.Timestamp(end)
    if start>end or start<calendar.min() or end>calendar.max():raise ValueError("回测范围超出已知市场日历。")
    start_pos=max(0,int(calendar.searchsorted(start))-int(warmup))
    dates=calendar[(calendar>=calendar[start_pos])&(calendar<=end)]
    pieces,files=[],[]
    for day in dates:
        path=root/"daily"/str(day.year)/f"{day:%Y%m%d}.csv"
        adj_path=root/"adj_factor"/str(day.year)/f"{day:%Y%m%d}.csv"
        raw=pd.read_csv(path,dtype={"ts_code":str,"trade_date":str})
        adj=pd.read_csv(adj_path,dtype={"ts_code":str,"trade_date":str})
        if raw.ts_code.duplicated().any() or adj.ts_code.duplicated().any():raise ValueError("行情/复权因子重复。")
        if not raw.trade_date.eq(day.strftime("%Y%m%d")).all() or not adj.trade_date.eq(day.strftime("%Y%m%d")).all():
            raise ValueError("日线文件与日期不一致。")
        frame=raw.merge(adj[["ts_code","adj_factor"]],on="ts_code",how="left",validate="one_to_one")
        frame=frame.rename(columns={"ts_code":"instrument"})
        frame["signal_date"]=day
        for col in ["open","high","low","close","pre_close","vol","amount","adj_factor"]:
            frame[col]=pd.to_numeric(frame[col],errors="coerce")
        frame["volume_shares"],frame["amount_yuan"]=frame.vol*100,frame.amount*1000
        pieces.append(frame[["signal_date","instrument","open","high","low","close","pre_close","volume_shares","amount_yuan","adj_factor"]])
        for p in [path,adj_path]:files.append(dict(path=str(p),sha256=library["stream_sha256"](p)))
    data=pd.concat(pieces,ignore_index=True).sort_values(["instrument","signal_date"])
    grouped=data.groupby("instrument",sort=False)
    data["adv20"]=grouped.amount_yuan.transform(lambda s:s.rolling(20,min_periods=10).mean())
    economic_price=data.close*data.adj_factor
    returns=economic_price.groupby(data.instrument,sort=False).pct_change(fill_method=None)
    data["sigma20"]=returns.groupby(data.instrument,sort=False).transform(lambda s:s.rolling(20,min_periods=10).std())
    data["previous_adj_factor"]=grouped.adj_factor.shift(1)
    data["adjustment_changed"]=(data.adj_factor.notna()&data.previous_adj_factor.notna()
                                &~np.isclose(data.adj_factor,data.previous_adj_factor,rtol=1e-9,atol=1e-12))
    data["market_observed"]=np.isfinite(data.close)&data.close.gt(0)&data.volume_shares.gt(0)
    data["historical_status_verified"]=False
    data["known_can_buy"],data["known_can_sell"]=False,False
    data["upper_limit"],data["lower_limit"]=np.nan,np.nan
    if status_path is not None:
        states=pd.read_csv(status_path,dtype={"instrument":str},parse_dates=["signal_date","available_at"])
        required={"signal_date","instrument","available_at","historical_status_verified","known_can_buy","known_can_sell","upper_limit","lower_limit"}
        if not required.issubset(states) or states.duplicated(["signal_date","instrument"]).any():raise ValueError("历史状态字段缺失或重复。")
        for name in ["historical_status_verified","known_can_buy","known_can_sell"]:
            if not states[name].isin([True,False]).all():raise ValueError("历史状态字段必须为布尔值。")
        latest=(pd.to_datetime(states.signal_date)+pd.Timedelta(hours=15)).map(library["as_shanghai"])
        supplied=states.available_at.map(library["as_shanghai"])
        if supplied.isna().any() or (supplied>latest).any():raise ValueError("状态在成交参考时点后才可用。")
        data=data.drop(columns=["historical_status_verified","known_can_buy","known_can_sell","upper_limit","lower_limit"])
        data=data.merge(states[list(required)],on=["signal_date","instrument"],how="left",validate="one_to_one")
        for name in ["historical_status_verified","known_can_buy","known_can_sell"]:data[name]=data[name].fillna(False).astype(bool)
        files.append(dict(path=str(status_path),sha256=library["stream_sha256"](status_path)))
    basic=root/"basic/stock_basic.csv"
    listing=pd.read_csv(basic,dtype=str,encoding="utf-8-sig")[["ts_code","list_date"]].rename(columns={"ts_code":"instrument"})
    listing["list_date"]=pd.to_datetime(listing.list_date,format="%Y%m%d",errors="coerce")
    data=data.merge(listing,on="instrument",how="left",validate="many_to_one")
    data=data.loc[data.instrument.str.endswith((".SH",".SZ")) & data.list_date.notna()].copy()
    files.append(dict(path=str(basic),sha256=library["stream_sha256"](basic)))
    for path in calendar_files:files.append(dict(path=str(path),sha256=library["stream_sha256"](path)))
    actions=pd.DataFrame(columns=["event_id","event_type","event_date","instrument"])
    if actions_path is not None:
        actions=pd.read_csv(actions_path,dtype={"instrument":str},parse_dates=["event_date"])
        required={"event_id","event_type","event_date","instrument"}
        if not required.issubset(actions) or actions.event_id.duplicated().any():raise ValueError("公司行动事件格式不合法。")
        if not actions.event_type.isin(["split","dividend_record","cash_settlement"]).all():raise ValueError("暂不支持的公司行动类型，拒绝忽略。")
        files.append(dict(path=str(actions_path),sha256=library["stream_sha256"](actions_path)))
    rules=pd.read_csv(rules_path) if rules_path is not None else None
    if rules is not None and ("verified" not in rules or not rules["verified"].isin([True,False]).all()):
        raise ValueError("规则表 verified 必须为明确布尔值。")
    if rules_path is not None:files.append(dict(path=str(rules_path),sha256=library["stream_sha256"](rules_path)))
    return dict(daily=data.set_index(["signal_date","instrument"]).sort_index(),calendar=calendar,
                actions=actions,rules=rules,files=files,source_hash=library["stable_hash"](files))


def market_day(market,day):
    try:return market["daily"].xs(pd.Timestamp(day),level="signal_date").copy()
    except KeyError:return market["daily"].iloc[:0].droplevel("signal_date").copy()


def known_market_for_decision(day_frame,day,library):
    known=day_frame.copy()
    known["price_as_of"]=library["as_shanghai"](pd.Timestamp(day)+pd.Timedelta(hours=15))
    known["as_of"]=known["price_as_of"]
    return known


def validate_backtest_config(config,source,market):
    if not config["models"] or set(config["models"])-set(source["models"]):raise ValueError("模型列表无效。")
    if len(config["models"])!=len(set(config["models"])):raise ValueError("模型列表重复。")
    if pd.Timestamp(config["start"])>pd.Timestamp(config["end"]):raise ValueError("回测起止日期无效。")
    if "report" in source:
        lower,upper=source["report"].signal_date.min(),source["report"].signal_date.max()
        if pd.Timestamp(config["start"])<lower or pd.Timestamp(config["end"])>upper:
            raise ValueError("回测区间超出信号覆盖范围，不能以全现金伪装无信号区间。")
    if config["initial_cash"]<=0 or not 0<config["volume_participation"]<=1:raise ValueError("资金/参与率配置无效。")
    if config["corporate_action_mode"] not in {"price_only","event_table"}:raise ValueError("公司行动模式无效。")
    if config["corporate_action_mode"]=="event_table" and not config.get("actions_coverage_verified",False):
        raise ValueError("公司行动模式要求显式确认事件覆盖完整性。")
    if config["mode"]=="strict":
        if source["metadata"]["result_level"]!="strict_status_check":raise ValueError("严格回测要求重新生成严格资格信号。")
        if not market["daily"].historical_status_verified.all() or market["rules"] is None:
            raise ValueError("严格模式缺少完整历史状态或板块规则。")
        if config["corporate_action_mode"]!="event_table" or not config.get("actions_coverage_verified",False):
            raise ValueError("严格模式缺少完整公司行动覆盖。")
    elif config["mode"]!="research":raise ValueError("mode 必须为 research 或 strict。")
    for name in ["commission_rate","min_commission","slippage_bps","impact_eta"]:
        if config[name]<0:raise ValueError("费用和执行损耗参数不能为负。")


In [ ]:
bt_market = load_backtest_market(
    bt_source, signal_lib, BT_CONFIG["start"], BT_CONFIG["end"],
    status_path=HISTORICAL_STATUS_PATH,
    actions_path=CORPORATE_ACTIONS_PATH,
    rules_path=HISTORICAL_QUANTITY_RULES_PATH,
)
validate_backtest_config(BT_CONFIG, bt_source, bt_market)
print(f"含预热的行情行数：{len(bt_market['daily']):,}")
display(bt_market["daily"].head(5))
print("当前按价格收益研究账本运行，公司行动缺口会单独写入审计。")


## 4. 现金与持仓主账本

**输入**：每个模型的起始现金、同一套组合配置及实际发生的事件。

账本持有现金总额、委托预留资金、实际股票数量、可卖与冻结数量，以及已确认应收款。始终核对：

```text
NAV = 现金总额 + 实际股票数量 × 最近有效估值 + 已确认应收款
现金总额 = 可用于交易的现金 + 委托预留资金
总股数 = 可卖股数 + 冻结股数
```

买入后股份进入冻结数量；只有进入新的市场交易日才解锁。重复调用同一天的开盘处理不会解锁新买股份。卖出只能减少可卖数量；实际成交才增减现金和持仓，提交委托只预留资金。

没有可观察行情的旧仓继续保留并标记估值陈旧，不因信号或行情缺行自动清仓。现金可用字段服务于交易，不等同于已经建模提款清算流程。

**输出**：`CashStockAccount`。实际账户快照直接传回已有组合接口，下一次决策使用真实成交后持仓。

In [ ]:
class CashStockAccount:
    """单模型、只做多账户；所有资金/股数变化都经过明确事件。"""
    def __init__(self,model,initial_cash,portfolio_config,library):
        self.model=model;self.library=library;self.portfolio_config=copy.deepcopy(portfolio_config)
        self.account_id=f"account__{model}"
        self.strategy_id=library["strategy_identity"](model,portfolio_config)
        self.initial_cash=float(initial_cash);self.cash_balance=float(initial_cash)
        self.positions={};self.pending={};self.reservations={};self.receivables={};self.entitlements={}
        self.current_day=None;self.applied_events=set()
        self.orders=[];self.fills=[];self.rejections=[];self.events=[];self.daily_records=[];self.position_records=[]

    @property
    def reserved_cash(self):return float(sum(self.reservations.values()))

    @property
    def cash_available(self):return self.cash_balance-self.reserved_cash

    @property
    def receivable_value(self):return float(sum(self.receivables.values()))

    @property
    def position_value(self):return float(sum(p["quantity"]*p["valuation_price"] for p in self.positions.values()))

    @property
    def nav(self):return self.cash_balance+self.position_value+self.receivable_value

    def assert_balanced(self):
        if self.cash_balance < -1e-6 or self.cash_available < -1e-6:raise AssertionError("现金或可用资金为负。")
        for position in self.positions.values():
            q,s,f=position["quantity"],position["sellable_quantity"],position["frozen_quantity"]
            if min(q,s,f)<0 or s+f!=q or any(int(x)!=x for x in [q,s,f]):raise AssertionError("股数账本不平衡。")
        if not np.isfinite(self.nav) or self.nav<=0:raise AssertionError("账户净值无效。")

    def start_day(self,day):
        day=pd.Timestamp(day)
        if self.current_day is not None and day<self.current_day:raise ValueError("账户时间不能回退。")
        if day==self.current_day:return
        self.current_day=day
        for position in self.positions.values():
            position["sellable_quantity"]=position["quantity"]
            position["frozen_quantity"]=0
        # 只有实际经过新的交易日，昨日新买股份才解锁。

    def snapshot(self,as_of):
        frame=pd.DataFrame.from_dict(self.positions,orient="index")
        required=["quantity","sellable_quantity","frozen_quantity","valuation_price","price_as_of"]
        if frame.empty:frame=pd.DataFrame(columns=required)
        frame.index.name="instrument"
        active=pd.DataFrame([dict(instrument=o["instrument"],status="submitted") for o in self.pending.values()])
        self.assert_balanced()
        return dict(account_id=self.account_id,strategy_id=self.strategy_id,model=self.model,
                    as_of=self.library["as_shanghai"](as_of),cash_available=self.cash_available,
                    cash_reserved=self.reserved_cash,nav=self.nav,other_assets=self.receivable_value,
                    positions=frame[required].copy(),open_orders=active,
                    snapshot_role="actual_event_account")

    def submit(self,order):
        if order["order_id"] in {o["order_id"] for o in self.orders}:raise ValueError("订单编号重复。")
        order=copy.deepcopy(order)
        if order["model"]!=self.model:raise ValueError("订单串用了其他模型账户。")
        if int(order["requested_quantity"])<=0:raise ValueError("委托数量必须为正。")
        order.update(status="submitted",filled_quantity=0,unfilled_quantity=int(order["requested_quantity"]))
        self.orders.append(order);self.pending[order["order_id"]]=order
        reserve=0.0
        if order["side"]=="buy" and not order["conditional_exit"]:
            spendable=max(0.0,self.cash_available-order["cash_buffer_amount"])
            reserve=min(spendable,float(order["estimated_cash"]))
        self.reservations[order["order_id"]]=reserve
        order["reserved_at_submission"]=reserve
        self.assert_balanced()

    def reject(self,order,day,reason,status="rejected"):
        self.reservations.pop(order["order_id"],None)
        self.pending.pop(order["order_id"],None)
        order["status"]=status;order["reason"]=reason
        order["unfilled_quantity"]=order["requested_quantity"]-order.get("filled_quantity",0)
        self.rejections.append(dict(order_id=order["order_id"],model=self.model,instrument=order["instrument"],
            event_date=pd.Timestamp(day),side=order["side"],requested_quantity=order["requested_quantity"],
            filled_quantity=order.get("filled_quantity",0),unfilled_quantity=order["unfilled_quantity"],reason=reason))
        self.assert_balanced()

    def apply_fill(self,order,day,quantity,price,costs):
        code=order["instrument"];quantity=int(quantity)
        self.reservations.pop(order["order_id"],None)
        notional=quantity*float(price)
        cash_cost=sum(float(costs[name]) for name in ["commission","stamp_tax","transfer_fee","execution_loss"])
        if quantity<=0:raise ValueError("成交量必须为正。")
        if order["side"]=="buy":
            if self.cash_balance-notional-cash_cost < self.reserved_cash-1e-7:raise AssertionError("成交超过可用现金。")
            self.cash_balance-=notional+cash_cost
            position=self.positions.setdefault(code,dict(quantity=0,sellable_quantity=0,frozen_quantity=0,
                valuation_price=float(price),price_as_of=self.library["as_shanghai"](pd.Timestamp(day)+pd.Timedelta(hours=15))))
            position["quantity"]+=quantity;position["frozen_quantity"]+=quantity
            position["valuation_price"]=float(price)
            position["price_as_of"]=self.library["as_shanghai"](pd.Timestamp(day)+pd.Timedelta(hours=15))
        else:
            position=self.positions.get(code)
            if position is None or quantity>position["sellable_quantity"]:raise AssertionError("卖出了不可卖股份。")
            if self.cash_balance+notional-cash_cost < self.reserved_cash-1e-7:raise AssertionError("卖出费用超过资金。")
            self.cash_balance+=notional-cash_cost
            position["quantity"]-=quantity;position["sellable_quantity"]-=quantity
            if position["quantity"]==0:del self.positions[code]
        order.update(status="filled" if quantity==order["requested_quantity"] else "partially_filled_then_expired",
                     filled_quantity=quantity,unfilled_quantity=order["requested_quantity"]-quantity)
        self.pending.pop(order["order_id"],None)
        record=dict(order_id=order["order_id"],model=self.model,instrument=code,side=order["side"],
                    signal_date=order["signal_date"],execution_date=pd.Timestamp(day),quantity=quantity,
                    reference_close=float(price),notional=notional,**costs)
        self.fills.append(record)
        if order["unfilled_quantity"]:
            self.rejections.append(dict(order_id=order["order_id"],model=self.model,instrument=code,
                event_date=pd.Timestamp(day),side=order["side"],requested_quantity=order["requested_quantity"],
                filled_quantity=quantity,unfilled_quantity=order["unfilled_quantity"],reason="remainder_expired_at_close"))
        self.assert_balanced()

    def mark_close(self,day,bars):
        for code,position in self.positions.items():
            if code in bars.index and bool(bars.loc[code,"market_observed"]) and np.isfinite(bars.loc[code,"close"]) and bars.loc[code,"close"]>0:
                position["valuation_price"]=float(bars.loc[code,"close"])
                position["price_as_of"]=self.library["as_shanghai"](pd.Timestamp(day)+pd.Timedelta(hours=15))
            stale=pd.Timestamp(position["price_as_of"]).date()!=pd.Timestamp(day).date()
            self.position_records.append(dict(model=self.model,date=pd.Timestamp(day),instrument=code,**position,
                                              stale_valuation=stale))
        self.assert_balanced()
        self.daily_records.append(dict(model=self.model,date=pd.Timestamp(day),cash_balance=self.cash_balance,
            cash_available=self.cash_available,cash_reserved=self.reserved_cash,market_value=self.position_value,
            receivables=self.receivable_value,nav=self.nav,position_count=len(self.positions),
            stale_position_count=sum(pd.Timestamp(p["price_as_of"]).date()!=pd.Timestamp(day).date() for p in self.positions.values())))


## 5. 公司行动接口与当前价格收益口径

默认 price_only 会记录持仓期间复权因子变化，但不会从一个复权比值猜测“现金分红还是送转”。因此该模式不能解释为完整总收益；审计记录会显示受影响的股票和日期。

event_table 模式支持以下明确事件：

| 事件 | 输入字段与处理 |
| --- | --- |
| split | `quantity_multiplier`；整股拆分改变数量与估值基准，调整尚未执行意图的数量与参考价；不支持的碎股处置会停止 |
| dividend_record | 在 event_date 登记日收盘，以实际持仓记录权益；另给 `ex_date, pay_date, net_cash_per_share` |
| 分红除息/到账 | 除息日确认应收款，到支付日才转为现金；登记后卖掉股票也保留已登记权益 |
| cash_settlement | 给 `cash_per_share`，按实际持仓做明确的现金处置；取消相关未完成委托 |

净分红需由输入明确给定，并披露税费假设；这里没有自动重建所有持有期差异化红利税。完整历史数据与税费核验仍是严格结果的前提。

**检查**：不得同时使用复权收益增加净值又重复增加现金分红。事件表若声称覆盖完整，而持仓复权变化找不到对应事件，运行会停止。

In [ ]:
def process_company_events(account,day,actions,phase):
    """可选事件表：登记权益、除息确认应收、到账、整股拆分和现金处置。"""
    day=pd.Timestamp(day)
    if phase=="open":
        for identifier,right in list(account.entitlements.items()):
            if right["ex_date"]==day and identifier not in account.receivables:
                account.receivables[identifier]=right["amount"]
                account.events.append(dict(date=day,instrument=right["instrument"],event="dividend_receivable",amount=right["amount"]))
            if right["pay_date"]==day:
                if identifier not in account.receivables:raise ValueError("分红未确认应收就试图到账。")
                amount=account.receivables.pop(identifier);account.cash_balance+=amount
                del account.entitlements[identifier]
                account.events.append(dict(date=day,instrument=right["instrument"],event="dividend_paid",amount=amount))
    if actions.empty:return
    todays=actions.loc[pd.to_datetime(actions.event_date).eq(day)]
    for event in todays.to_dict("records"):
        kind=event["event_type"]
        if (kind=="dividend_record")!=(phase=="record_close"):continue
        identifier=str(event["event_id"])
        if identifier in account.applied_events:raise ValueError("公司行动重复处理。")
        account.applied_events.add(identifier)
        code=event["instrument"];position=account.positions.get(code)
        if position is None:continue
        if kind=="dividend_record":
            ex_date,pay_date=pd.Timestamp(event["ex_date"]),pd.Timestamp(event["pay_date"])
            if not day<ex_date<=pay_date:raise ValueError("分红登记、除息和到账日期无效。")
            net=float(event["net_cash_per_share"])
            if net<0 or not np.isfinite(net):raise ValueError("需提供明确的非负净分红假设。")
            account.entitlements[identifier]=dict(instrument=code,amount=position["quantity"]*net,
                ex_date=ex_date,pay_date=pay_date,record_quantity=position["quantity"])
            account.events.append(dict(date=day,instrument=code,event="dividend_right_recorded",quantity=position["quantity"]))
        elif kind=="split":
            factor=float(event["quantity_multiplier"])
            quantities=[position[name]*factor for name in ["quantity","sellable_quantity","frozen_quantity"]]
            if factor<=0 or not all(np.isclose(q,round(q)) for q in quantities):raise ValueError("需单独提供零碎股处置，拒绝隐式取整拆分。")
            for name,quantity in zip(["quantity","sellable_quantity","frozen_quantity"],quantities):position[name]=int(round(quantity))
            position["valuation_price"]/=factor
            for order in account.pending.values():
                if order["instrument"]==code:
                    quantity=order["requested_quantity"]*factor
                    if not np.isclose(quantity,round(quantity)):raise ValueError("拆分后的待执行数量不是整股。")
                    order["requested_quantity"]=int(round(quantity));order["unfilled_quantity"]=int(round(quantity))
                    order["reference_price"]/=factor
                    order["split_adjusted"]=True
            account.events.append(dict(date=day,instrument=code,event="split",factor=factor))
        elif kind=="cash_settlement":
            cash=position["quantity"]*float(event["cash_per_share"])
            if cash<0 or not np.isfinite(cash):raise ValueError("现金处置金额无效。")
            for order in list(account.pending.values()):
                if order["instrument"]==code:account.reject(order,day,"company_cash_settlement",status="cancelled")
            account.cash_balance+=cash;del account.positions[code]
            account.events.append(dict(date=day,instrument=code,event="cash_settlement",amount=cash))
        else:raise ValueError(f"未知公司行动：{kind}")
    account.assert_balanced()


## 6. 费用、涨跌停与执行损耗

**输入**：买卖方向、实际成交数量、参考价格、成交日及信号日已经知道的 ADV/波动。

默认历史费率表限定2022—2025年：卖出印花税在2023-08-28前后切换，过户费在2022-04-29前后切换。佣金和最低费用来自本实验配置；如果佣金已包含经手等费用，不再重复加扣。

冲击率采用 `eta × sigma20 × sqrt(成交金额 / 已知ADV20)` 的研究近似，系数尚未按本项目执行数据校准。滑点和冲击作为额外执行损耗单列，原始收盘价仍是参考记账价。

涨跌停优先使用已核验的历史限制价格。research 模式缺历史状态时按板块作显式假设，不能完整覆盖ST、IPO和所有特殊交易日；strict 模式缺真实限制价就停止。买入碰涨停与卖出碰跌停分别处理，不把两个方向一起禁止。

**输出**：分项费用、限制价与规则来源。默认规则扩展到2026年等新时期前需要更新并核验相应历史规则。

In [ ]:
def transaction_costs(side,quantity,price,day,config,adv20=np.nan,sigma20=np.nan):
    notional=float(quantity)*float(price)
    if notional<=0:return dict(commission=0.0,stamp_tax=0.0,transfer_fee=0.0,execution_loss=0.0,impact_rate=0.0)
    day=pd.Timestamp(day)
    if day<pd.Timestamp("2022-01-01") or day>pd.Timestamp("2025-12-31"):
        raise ValueError("默认历史费率范围为2022—2025，扩展日期前请更新规则表。")
    stamp_rate=0.001 if day<pd.Timestamp("2023-08-28") else 0.0005
    transfer_rate=0.00002 if day<pd.Timestamp("2022-04-29") else 0.00001
    if config.get("zero_cost_test",False):return dict(commission=0.0,stamp_tax=0.0,transfer_fee=0.0,execution_loss=0.0,impact_rate=0.0)
    commission=max(notional*config["commission_rate"],config["min_commission"])
    impact=0.0
    if config["impact_eta"] and np.isfinite(adv20) and adv20>0 and np.isfinite(sigma20):
        impact=config["impact_eta"]*max(0.0,sigma20)*np.sqrt(notional/adv20)
    return dict(commission=commission,stamp_tax=notional*stamp_rate if side=="sell" else 0.0,
                transfer_fee=notional*transfer_rate,execution_loss=notional*(config["slippage_bps"]/10000+impact),impact_rate=impact)


def cash_charge(costs):
    return sum(costs[name] for name in ["commission","stamp_tax","transfer_fee","execution_loss"])


def board_of(code):
    if code.endswith(".SH") and code.startswith(("688","689")):return "star"
    if code.endswith(".SZ") and code.startswith(("300","301")):return "gem"
    return "main"


def historical_price_limits(bar,code,mode):
    verified=bool(bar.get("historical_status_verified",False))
    upper,lower=bar.get("upper_limit",np.nan),bar.get("lower_limit",np.nan)
    if verified and np.isfinite(upper) and np.isfinite(lower):return float(upper),float(lower),"verified"
    if mode=="strict":raise ValueError("缺少已核验的历史涨跌停价。")
    previous=bar.get("pre_close",np.nan)
    if not np.isfinite(previous) or previous<=0:return np.nan,np.nan,"unknown"
    rate=Decimal("0.20") if board_of(code) in {"star","gem"} else Decimal("0.10")
    base=Decimal(str(previous))
    upper=float((base*(1+rate)).quantize(Decimal("0.01"),rounding=ROUND_HALF_UP))
    lower=float((base*(1-rate)).quantize(Decimal("0.01"),rounding=ROUND_HALF_UP))
    return upper,lower,"research_board_assumption"


## 7. 收盘后生成下一交易日订单

**输入**：t 日信号、t 日已知行情、当前真实账户和组合规则。

依次调用上一模块的目标权重与股数意图接口，然后生成带有提交时间、下一执行日、方向、数量和资金条件的订单。数量依据 t 日价格与NAV确定，不用 t+1 最终收盘价预先“精确算好”股数。

对依赖旧仓退出的新买入，预先保存候选与按 t 日计算的数量，标为条件买单；不预留并不存在的卖出所得。真正执行时，必须先看旧仓是否完整退出、是否有空席位以及真实可用资金。执行期间不重新选股。

规划容量取 t 日已经知道的 ADV，日后实际成交量限制只影响成交，不会反过来重排昨天的候选。

**输出**：目标记录、原因日志和待执行订单。不同模型、日期、方向、股票的订单编号可重复生成，便于核对。

In [ ]:
def plan_next_close_orders(account,day,signals,bars,market,portfolio_config,config):
    library=account.library
    decision_at=library["as_shanghai"](signals.signal_available_at.max())
    known=known_market_for_decision(bars,day,library)
    snapshot=account.snapshot(decision_at)
    targets,decisions=library["build_target_weights"](signals,snapshot,known,portfolio_config)
    quantities,checks=library["estimate_target_shares"](targets,snapshot,known,market["rules"],portfolio_config)
    if targets.empty:return targets,decisions,[]
    orders=[]
    def make_order(row,side,quantity,conditional=False):
        code=row["instrument"]
        bar=bars.loc[code]
        reference=float(bar["close"])
        rule=library["rule_for_instrument"](code,pd.Timestamp(day),market["rules"],config["mode"])
        if rule is None:return
        # 订单规划只使用信号日的 ADV 和波动；不读取执行日价格与成交量。
        adv=float(bar.get("adv20",np.nan))
        sigma=float(bar.get("sigma20",np.nan))
        if side=="buy" and config.get("planning_adv_limit",0)>0:
            if not np.isfinite(adv) or adv<=0:return
            allowed=adv*config["planning_adv_limit"]/reference
            quantity=min(quantity,library["legal_order_quantity"](allowed,int(rule["buy_min"]),int(rule["buy_step"])))
        if quantity<=0:return
        estimated=transaction_costs(side,quantity,reference,day,config,adv,sigma)
        identifier=library["stable_hash"](dict(account=account.account_id,policy=account.strategy_id,
            signal_date=str(pd.Timestamp(day).date()),instrument=code,side=side,conditional=conditional))[:24]
        orders.append(dict(order_id=identifier,model=account.model,strategy_id=account.strategy_id,
            instrument=code,side=side,signal_date=pd.Timestamp(day),submitted_at=decision_at,
            execution_date=pd.Timestamp(row["earliest_execution_date"]),requested_quantity=int(quantity),
            original_requested_quantity=int(quantity),reference_price=reference,
            reference_price_as_of=library["as_shanghai"](pd.Timestamp(day)+pd.Timedelta(hours=15)),
            conditional_exit=bool(conditional),top_k=int(portfolio_config["top_k"]),
            cash_buffer_amount=(1-portfolio_config["target_stock_weight"])*snapshot["nav"],
            estimated_cash=quantity*reference+cash_charge(estimated) if side=="buy" else 0.0,
            adv20_known=adv,sigma20_known=sigma,reason=row["decision_reason"],status="planned"))
    for row in quantities.to_dict("records"):
        delta=int(row["quantity_delta"])
        if delta:make_order(row,"buy" if delta>0 else "sell",abs(delta))
    # 条件买入仍按 t 日价格预先确定数量；只有真实卖出后才可能激活。
    for row in targets.loc[targets.decision_reason.eq("requires_exit_fill")].to_dict("records"):
        code=row["instrument"]
        if code not in bars.index:continue
        rule=library["rule_for_instrument"](code,pd.Timestamp(day),market["rules"],config["mode"])
        if rule is None:continue
        raw=row["desired_weight"]*snapshot["nav"]/float(bars.loc[code,"close"])
        quantity=library["legal_order_quantity"](raw,int(rule["buy_min"]),int(rule["buy_step"]))
        make_order(row,"buy",quantity,True)
    return targets,decisions,orders


## 8. 收盘撮合：把意图转为实际成交

到 t+1 收盘，才读取当天参考价格与实际日成交量。撮合按固定顺序处理：卖单优先，再处理已有买单和条件买单。

每笔订单检查：时间先后、行情与状态、买卖方向的涨跌停、T+1、数量规则、实际剩余容量、席位和资金。价格上涨或费用增加后资金不足，会缩量或拒单；不会产生负现金。

一次日频订单只在这个收盘参考窗口聚合成交一次，剩余部分当日失效，记录为 `partially_filled_then_expired`。未成交部分不自动滚动为下一日订单；下一次策略使用真实账户重新决定。只有实际成交才产生费用和持仓。

**限制说明**：1%的日成交量容量是日线代理，不是集合竞价排队模型；交易单位和单笔数量还会再次复查。真实盘口、暂停临停和复杂交易时段需要未来更细数据适配。

**输出**：成交、拒单、部分成交和剩余取消记录。

In [ ]:
def execute_close_orders(account,day,bars,market,config):
    library=account.library;day=pd.Timestamp(day)
    used=defaultdict(int);closed_positions=set()
    due=[o for o in account.pending.values() if pd.Timestamp(o["execution_date"])<=day]
    due.sort(key=lambda o:(0 if o["side"]=="sell" else 1,int(o["conditional_exit"]),o["instrument"],o["order_id"]))
    for order in due:
        identifier,code=order["order_id"],order["instrument"]
        account.reservations.pop(identifier,None)
        if pd.Timestamp(order["execution_date"])!=day:
            account.reject(order,day,"missed_execution_date",status="expired");continue
        if library["as_shanghai"](order["submitted_at"])>=library["as_shanghai"](day+pd.Timedelta(hours=15)):
            account.reject(order,day,"not_submitted_before_execution");continue
        if code not in bars.index:
            account.reject(order,day,"missing_market_bar");continue
        bar=bars.loc[code]
        if not bool(bar["market_observed"]):
            account.reject(order,day,"no_observed_liquidity");continue
        if bool(bar.get("historical_status_verified",False)):
            flag="known_can_buy" if order["side"]=="buy" else "known_can_sell"
            if not bool(bar.get(flag,False)):
                account.reject(order,day,"verified_trading_restriction");continue
        elif config["mode"]=="strict":
            account.reject(order,day,"historical_status_unknown");continue
        upper,lower,limit_source=historical_price_limits(bar,code,config["mode"])
        price=float(bar["close"])
        if not np.isfinite(upper) or not np.isfinite(lower):
            account.reject(order,day,"price_limit_unknown");continue
        if order["side"]=="buy" and price>=upper-1e-8:
            account.reject(order,day,"limit_up_buy_blocked");continue
        if order["side"]=="sell" and price<=lower+1e-8:
            account.reject(order,day,"limit_down_sell_blocked");continue
        if order["conditional_exit"] and (not closed_positions or len(account.positions)>=order["top_k"]):
            account.reject(order,day,"no_real_exit_or_no_free_slot");continue
        if order["side"]=="buy" and code not in account.positions and len(account.positions)>=order["top_k"]:
            account.reject(order,day,"actual_holding_slots_full");continue
        rule=library["rule_for_instrument"](code,day,market["rules"],config["mode"])
        if rule is None:
            account.reject(order,day,"quantity_rule_unknown");continue
        maximum_order=100000 if board_of(code)=="star" else 1000000
        capacity=max(0,int(float(bar["volume_shares"])*config["volume_participation"])-used[code])
        allowed=min(int(order["requested_quantity"]),capacity,maximum_order)
        if order["side"]=="sell":
            position=account.positions.get(code)
            sellable=0 if position is None else int(position["sellable_quantity"])
            allowed=min(allowed,sellable)
            if position is not None and allowed>=position["quantity"]:
                quantity=int(position["quantity"])
            else:quantity=library["legal_order_quantity"](allowed,int(rule["sell_min"]),int(rule["sell_step"]))
        else:quantity=library["legal_order_quantity"](allowed,int(rule["buy_min"]),int(rule["buy_step"]))
        if quantity<=0:
            account.reject(order,day,"sellable_volume_or_lot_limit");continue
        def costs_for(q):return transaction_costs(order["side"],q,price,day,config,order["adv20_known"],order["sigma20_known"])
        costs=costs_for(quantity)
        if order["side"]=="buy":
            available=max(0.0,account.cash_available-order["cash_buffer_amount"])
            if quantity*price+cash_charge(costs)>available:
                rate=1+config["commission_rate"]+config["slippage_bps"]/10000+costs["impact_rate"]+0.00002
                raw=max(0.0,(available-config["min_commission"])/(price*rate))
                quantity=min(quantity,library["legal_order_quantity"](raw,int(rule["buy_min"]),int(rule["buy_step"])))
                costs=costs_for(quantity)
                while quantity>0 and quantity*price+cash_charge(costs)>available+1e-8:
                    quantity=library["legal_order_quantity"](quantity-int(rule["buy_step"]),int(rule["buy_min"]),int(rule["buy_step"]))
                    costs=costs_for(quantity)
            if quantity<=0:
                account.reject(order,day,"actual_cash_or_fee_limit");continue
        elif account.cash_available+quantity*price-cash_charge(costs)<-1e-8:
            account.reject(order,day,"sell_fee_cash_shortfall");continue
        existed=code in account.positions
        account.apply_fill(order,day,quantity,price,costs)
        order["limit_source"]=limit_source
        if order["side"]=="sell" and existed and code not in account.positions:closed_positions.add(code)
        used[code]+=quantity
    account.assert_balanced()


## 9. 事件循环与账户连续性

每天按照同一顺序运行：

1. 进入下一个交易日，解锁此前买入股份。
2. 审计复权变化，按配置处理开盘公司行动和应收到账。
3. 执行此前已提交、今天到期的订单。
4. 按当前可观察收盘价估值，记录现金、股数、费用与净值。
5. 在登记日收盘记录分红权益。
6. 收盘后读取今天信号，生成下一交易日订单并预留资金。

信号按月份缓存读取，账户不会在月份或滚动模型窗口切换时清零。回测结束不强制按最后价格卖光，只取消未执行委托，并保留期末持仓和应收款。

**输出**：每个模型的完整账本结果。价格变化造成的仓位超限单独记录；不靠修改目标表假装已经减仓。

In [ ]:
def run_one_model(model,source,market,library,config,output_dir=None,signal_frames=None):
    portfolio=copy.deepcopy(config["portfolio"])
    portfolio.update(status_mode=config["mode"],commission_rate_estimate=config["commission_rate"],
                     min_commission_estimate=config["min_commission"])
    account=CashStockAccount(model,config["initial_cash"],portfolio,library)
    days=market["calendar"][(market["calendar"]>=pd.Timestamp(config["start"]))&(market["calendar"]<=pd.Timestamp(config["end"]))]
    if len(days)<2:raise ValueError("回测至少需要两个交易日。")
    previous_month=None;month_data=pd.DataFrame()
    targets_all,decisions_all,action_audit=[],[],[]
    for day in days:
        account.start_day(day)
        bars=market_day(market,day)
        for code,position in list(account.positions.items()):
            if code in bars.index and bool(bars.loc[code,"adjustment_changed"]):
                covered=False
                if not market["actions"].empty:
                    events=market["actions"]
                    covered=bool((events.instrument.eq(code)&pd.to_datetime(events.event_date).eq(day)&events.event_type.eq("split")).any())
                covered |= any(right["instrument"]==code and right["ex_date"]==day for right in account.entitlements.values())
                if not covered:
                    action_audit.append(dict(date=day,instrument=code,model=model,quantity=position["quantity"],
                        previous_adj_factor=bars.loc[code,"previous_adj_factor"],adj_factor=bars.loc[code,"adj_factor"],
                        reason="unmodeled_adjustment_in_price_only_account"))
                    if config["corporate_action_mode"]=="event_table":
                        raise ValueError(f"持仓复权因子变化缺少对应事件：{day.date()} {code}")
        if config["corporate_action_mode"]=="event_table":process_company_events(account,day,market["actions"],"open")
        pretrade=account.cash_balance+account.receivable_value+sum(
            p["quantity"]*(float(bars.loc[c,"close"]) if c in bars.index and np.isfinite(bars.loc[c,"close"]) and bars.loc[c,"close"]>0 else p["valuation_price"])
            for c,p in account.positions.items())
        fills_before=len(account.fills)
        execute_close_orders(account,day,bars,market,config)
        account.mark_close(day,bars)
        account.daily_records[-1]["nav_pretrade"]=pretrade
        account.daily_records[-1]["gross_traded_notional"]=sum(f["notional"] for f in account.fills[fills_before:])
        if config["corporate_action_mode"]=="event_table":process_company_events(account,day,market["actions"],"record_close")
        # 收盘后才读取同日信号。信号中的未来标签从不传入策略。
        if signal_frames is not None:
            signals=signal_frames.loc[signal_frames.signal_date.eq(day)].copy()
        else:
            month=day.strftime("%Y%m")
            if month!=previous_month:
                month_data=library["load_signal_data"](source["run"],model=model,
                    start=max(pd.Timestamp(config["start"]),day.replace(day=1)),
                    end=min(pd.Timestamp(config["end"]),day+pd.offsets.MonthEnd(0)))
                previous_month=month
            signals=month_data.loc[month_data.signal_date.eq(day)].copy() if len(month_data) else pd.DataFrame()
        if len(signals):
            if set(signals.model)!={model} or library["FORBIDDEN_DECISION_FIELDS"].intersection(signals.columns):
                raise ValueError("回测信号混用模型或含未来标签。")
            executable=signals.earliest_execution_date.notna()&signals.earliest_execution_date.le(pd.Timestamp(config["end"]))
            if executable.any():
                targets,decisions,orders=plan_next_close_orders(account,day,signals,bars,market,portfolio,config)
                if len(targets):targets_all.append(targets)
                if len(decisions):decisions_all.append(decisions.assign(signal_date=day,model=model))
                for order in orders:account.submit(order)
        # 预留资金属于现金的一部分，提交委托不改变总 NAV。
        account.daily_records[-1].update(cash_available=account.cash_available,cash_reserved=account.reserved_cash)
        if config.get("progress_every",20) and len(account.daily_records)%config["progress_every"]==0:
            print(f"  {model}: {day.date()}，已处理 {len(account.daily_records)}/{len(days)} 个交易日",flush=True)
    for order in list(account.pending.values()):account.reject(order,days[-1],"end_of_test_pending_cancelled",status="cancelled")
    account.daily_records[-1].update(cash_available=account.cash_available,cash_reserved=account.reserved_cash)
    daily=pd.DataFrame(account.daily_records).set_index("date")
    daily["daily_return"]=daily.nav.pct_change(fill_method=None)
    daily.iloc[0,daily.columns.get_loc("daily_return")]=daily.nav.iloc[0]/account.initial_cash-1
    daily["normalized_nav"]=daily.nav/account.initial_cash
    daily["drawdown"]=daily.nav/daily.nav.cummax()-1
    daily["gross_turnover"]=daily.gross_traded_notional/daily.nav_pretrade
    daily["one_way_turnover"]=daily.gross_turnover/2
    results=dict(daily=daily,orders=pd.DataFrame(account.orders),fills=pd.DataFrame(account.fills),
        rejections=pd.DataFrame(account.rejections),positions=pd.DataFrame(account.position_records),
        company_events=pd.DataFrame(account.events),company_action_audit=pd.DataFrame(action_audit),
        targets=pd.concat(targets_all,ignore_index=True) if targets_all else pd.DataFrame(),
        decisions=pd.concat(decisions_all,ignore_index=True) if decisions_all else pd.DataFrame(),
        terminal_snapshot=account.snapshot(library["as_shanghai"](days[-1]+pd.Timedelta(hours=15,minutes=30))),
        pending_dividend_rights=copy.deepcopy(account.entitlements))
    positions=results["positions"]
    if len(positions):
        positions["position_weight"]=positions.quantity*positions.valuation_price/positions.date.map(daily.nav)
        breaches=positions.loc[positions.position_weight.gt(portfolio["single_name_weight_cap"]+1e-10)].copy()
        breaches["reason"]="mark_to_market_single_name_breach"
    else:breaches=pd.DataFrame()
    results["constraint_breaches"]=breaches
    if output_dir is not None:save_model_backtest(results,output_dir,library)
    return results


## 10. 报告、基准与版本化保存

净值来自账户的现金、实际持仓和应收款。日收益、回撤和换手均由账本计算。实际成交金额/NAV是双边金额换手，其一半另列为单边口径，不能与替换股票数量混淆。

基准使用现有沪深300**价格指数**，在相同首日收盘归一化为1；现金对照为1。它是无成本指数参考，不是已经加入相同费用的可交易指数复制策略。全收益基准与同池等权可交易对照需要后续补齐。

Sharpe 使用日超额收益的均值/标准差，年化常数默认252；首个收盘为比较起点。短区间年化数值可能非常不稳定，不能据此挑选最优模型。

保存配置、源文件哈希、订单、成交、持仓、现金、目标、拒单、公司行动缺口和期末快照。只有所有指定模型成功才更新该目录的 latest；失败保留失败状态，旧成功指针保持不变。

**输出**：`run_backtest` 与 `load_backtest_run`，可统一运行和校验读取。

In [ ]:
def performance_summary(results,initial_cash,benchmark=None,annual_days=252,risk_free_annual=0.0):
    daily=results["daily"];returns=daily.daily_return.iloc[1:]
    periods=len(returns)
    total=float(daily.nav.iloc[-1]/initial_cash-1)
    annual=float((1+total)**(annual_days/periods)-1) if periods>0 and total>-1 else np.nan
    std=float(returns.std(ddof=1)) if periods>1 else np.nan
    rf=(1+risk_free_annual)**(1/annual_days)-1
    sharpe=float((returns.mean()-rf)/std*np.sqrt(annual_days)) if np.isfinite(std) and std>0 else np.nan
    summary=dict(model=daily.model.iloc[0],trading_days=len(daily),return_intervals=periods,total_return=total,
        annualized_return=annual,annualized_volatility=std*np.sqrt(annual_days),sharpe=sharpe,
        max_drawdown=float(daily.drawdown.min()),terminal_nav=float(daily.nav.iloc[-1]),
        terminal_cash=float(daily.cash_balance.iloc[-1]),terminal_market_value=float(daily.market_value.iloc[-1]),
        terminal_receivables=float(daily.receivables.iloc[-1]),mean_gross_turnover=float(daily.gross_turnover.mean()),
        stale_position_days=int(daily.stale_position_count.sum()),orders=len(results["orders"]),
        fills=len(results["fills"]),unmodeled_adjustment_events=len(results["company_action_audit"]))
    fills=results["fills"]
    for name in ["commission","stamp_tax","transfer_fee","execution_loss"]:
        summary[name]=float(fills[name].sum()) if len(fills) else 0.0
    summary["total_trading_cost"]=sum(summary[n] for n in ["commission","stamp_tax","transfer_fee","execution_loss"])
    if len(results["orders"]):
        requested=float(results["orders"].requested_quantity.sum())
        filled=float(results["orders"].filled_quantity.sum())
        summary["share_fill_ratio"]=filled/requested if requested else np.nan
    if benchmark is not None:
        aligned=benchmark.reindex(daily.index)
        if aligned.isna().any():raise ValueError("基准交易日缺失，拒绝前填或跳过。")
        relative=(daily.normalized_nav/aligned)
        active=returns-aligned.pct_change(fill_method=None).iloc[1:]
        active_std=active.std(ddof=1)
        summary.update(benchmark_total_return=float(aligned.iloc[-1]-1),
            relative_total_return=float(relative.iloc[-1]-1),
            information_ratio=float(active.mean()/active_std*np.sqrt(annual_days)) if active_std>0 else np.nan)
    return summary


def load_price_benchmark(project_dir,dates):
    paths=sorted((Path(project_dir)/"data/raw/benchmark").glob("csi300_*.csv"))
    if not paths:raise FileNotFoundError("缺少沪深300价格指数。")
    pieces=[pd.read_csv(path,dtype={"trade_date":str})[["trade_date","close"]] for path in paths]
    frame=pd.concat(pieces).drop_duplicates()
    if frame.trade_date.duplicated().any():raise ValueError("重复基准数据的价格不一致。")
    frame["date"]=pd.to_datetime(frame.trade_date,format="%Y%m%d")
    series=frame.set_index("date")["close"].reindex(dates)
    if series.isna().any():raise ValueError("沪深300基准日期缺失。")
    return series/series.iloc[0]


def save_model_backtest(results,output_dir,library):
    root=Path(output_dir);root.mkdir(parents=True,exist_ok=True)
    mapping={"daily":"account.parquet","orders":"orders.parquet","fills":"fills.parquet",
             "rejections":"rejections.parquet","positions":"positions.parquet","targets":"target_positions.parquet",
             "decisions":"decisions.parquet","company_events":"company_events.parquet",
             "company_action_audit":"unmodeled_company_actions.parquet","constraint_breaches":"constraint_breaches.parquet"}
    for key,filename in mapping.items():
        frame=results[key]
        if frame.empty and not len(frame.columns):frame=pd.DataFrame({"empty":pd.Series(dtype="bool")})
        frame.to_parquet(root/filename,index=key=="daily")
    results["daily"][["normalized_nav","daily_return","drawdown","gross_turnover","one_way_turnover"]].to_parquet(root/"daily_returns.parquet")
    snapshot=results["terminal_snapshot"].copy()
    snapshot["positions"]=snapshot["positions"].reset_index().to_dict("records")
    snapshot["open_orders"]=snapshot["open_orders"].to_dict("records")
    library["save_json_atomic"](root/"terminal_snapshot.json",snapshot)
    library["save_json_atomic"](root/"pending_dividend_rights.json",results["pending_dividend_rights"])


def run_backtest(source,market,library,config,output_root):
    validate_backtest_config(config,source,market)
    root=Path(output_root)
    stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run=root/f"run_{stamp}_{uuid.uuid4().hex[:8]}";run.mkdir(parents=True,exist_ok=False)
    level="research_price_return" if config["corporate_action_mode"]=="price_only" else "research_event_cash_account"
    if config["mode"]=="strict":level="strict_inputs_daily_execution_proxy"
    meta=dict(schema=BACKTEST_SCHEMA,status="running",models=config["models"],result_level=level,
        config_hash=library["stable_hash"](config),source_hash=source["source_hash"],market_hash=market["source_hash"],
        signal_run=str(source["run"]),corporate_action_mode=config["corporate_action_mode"],
        includes_supplied_company_events=config["corporate_action_mode"]=="event_table",
        economic_total_return_independently_validated=False,
        end_liquidation=False,cash_interest=0.0)
    library["save_json_atomic"](run/"config.json",config)
    library["save_json_atomic"](run/"source.json",dict(signals=source["manifest"],market_files=market["files"]))
    library["save_json_atomic"](run/"metadata.json",meta)
    reports=[];daily_curves=[]
    try:
        for model in config["models"]:
            print(f"开始回测 {model}：{config['start']} ~ {config['end']}",flush=True)
            results=run_one_model(model,source,market,library,config,run/model)
            benchmark=load_price_benchmark(source["project_dir"],results["daily"].index)
            report=performance_summary(results,config["initial_cash"],benchmark,config["annual_days"],config["risk_free_annual"])
            report["result_level"]=level;reports.append(report)
            daily_curves.append(results["daily"][["normalized_nav"]].rename(columns={"normalized_nav":model}))
            benchmark.rename("csi300_price_index").to_frame().to_parquet(run/"benchmark.parquet")
            pd.DataFrame(reports).to_csv(run/"performance_summary.csv",index=False)
            print(f"  {model} 完成：{report['orders']} 笔委托，{report['fills']} 笔聚合成交；"
                  f"未建模复权事件 {report['unmodeled_adjustment_events']} 条。",flush=True)
        curves=pd.concat(daily_curves,axis=1)
        curves["cash"]=1.0;curves["csi300_price_index"]=benchmark
        curves.to_parquet(run/"nav_comparison.parquet")
        inventory={str(p.relative_to(run)):library["stream_sha256"](p) for p in run.rglob("*") if p.is_file() and p.name!="metadata.json"}
        meta.update(status="complete",files_sha256=inventory,completed_at=datetime.now(timezone.utc).isoformat())
        library["save_json_atomic"](run/"metadata.json",meta)
        library["save_json_atomic"](root/"latest_run.json",{"run_dir":run.name})
    except BaseException as exc:
        meta.update(status="failed",error=f"{type(exc).__name__}: {exc}")
        library["save_json_atomic"](run/"metadata.json",meta)
        raise
    return run,pd.DataFrame(reports),curves


def load_backtest_run(run_dir,library,verify_hashes=True):
    run=Path(run_dir);meta=library["read_json_file"](run/"metadata.json")
    if meta.get("status")!="complete" or meta.get("schema")!=BACKTEST_SCHEMA:raise ValueError("回测运行未成功完成。")
    if verify_hashes:
        for relative,digest in meta["files_sha256"].items():
            if library["stream_sha256"](run/relative)!=digest:raise ValueError(f"回测产物校验失败：{relative}")
    return dict(run_dir=run,metadata=meta,config=library["read_json_file"](run/"config.json"),
                summary=pd.read_csv(run/"performance_summary.csv"),curves=pd.read_parquet(run/"nav_comparison.parquet"))


## 11. 先做可手工核对的自动验收

这里的合成场景检查账户守恒、T+1、部分成交、资金复查、涨跌停、费用生效日、分红权益与到账、拆分、缺行情保留持仓、无成本恒价回放以及未来数据隔离。

测试使用自己的小账户，不读取真实测试收益决定是否通过。恒定价格且零费用时换仓不能凭空赚钱；变更未来信号不能改变过去订单。严格模式缺输入必须明确失败。

**输出**：逐项 pass；有任何异常先修复，再运行真实回放。

In [ ]:
def run_backtest_acceptance(library):
    rows=[]
    def check(name,condition):
        if not condition:raise AssertionError(name)
        rows.append(dict(check=name,status="pass"))
    def rejects(name,function):
        try:function()
        except (ValueError,AssertionError):rows.append(dict(check=name,status="pass"));return
        raise AssertionError(name)
    config=default_backtest_config();config.update(progress_every=0,impact_eta=0.0,planning_adv_limit=0.0)
    portfolio=copy.deepcopy(config["portfolio"])
    portfolio.update(status_mode="research",commission_rate_estimate=config["commission_rate"],min_commission_estimate=config["min_commission"])
    day=pd.Timestamp("2025-01-03");code="000001.SZ"
    def order(identifier,side,quantity,execution=day,conditional=False):
        return dict(order_id=identifier,model="ols",strategy_id=library["strategy_identity"]("ols",portfolio),
            instrument=code,side=side,signal_date=pd.Timestamp("2025-01-02"),
            submitted_at=library["as_shanghai"]("2025-01-02 15:30"),execution_date=pd.Timestamp(execution),
            requested_quantity=quantity,original_requested_quantity=quantity,reference_price=10.0,
            reference_price_as_of=library["as_shanghai"]("2025-01-02 15:00"),conditional_exit=conditional,
            cash_buffer_amount=0.0,estimated_cash=quantity*10+5 if side=="buy" else 0.0,
            top_k=50,adv20_known=1e8,sigma20_known=.02,reason="test")
    def bar(close=10.0,volume=1e6,upper=11.0,lower=9.0,**kwargs):
        return pd.DataFrame([dict(instrument=code,close=close,pre_close=10.0,volume_shares=volume,
            market_observed=volume>0,historical_status_verified=True,known_can_buy=True,known_can_sell=True,
            upper_limit=upper,lower_limit=lower,**kwargs)]).set_index("instrument")
    market=dict(rules=None)
    account=CashStockAccount("ols",10000,portfolio,library);account.start_day(day)
    account.submit(order("buy","buy",100))
    execute_close_orders(account,day,bar(),market,config)
    check("buy_cash_and_position_conservation",np.isclose(account.nav,10000-cash_charge(account.fills[0])))
    check("new_buy_is_locked",account.positions[code]["quantity"]==100 and account.positions[code]["sellable_quantity"]==0)
    account.start_day(day)
    check("same_day_does_not_unlock",account.positions[code]["sellable_quantity"]==0)
    account.submit(order("sell_t0","sell",100))
    execute_close_orders(account,day,bar(),market,config)
    check("t_plus_zero_sell_rejected",account.positions[code]["quantity"]==100 and len(account.fills)==1)
    account.start_day(pd.Timestamp("2025-01-06"))
    check("next_trading_day_unlocks",account.positions[code]["sellable_quantity"]==100)
    account.submit(order("sell_t1","sell",100,execution="2025-01-06"))
    execute_close_orders(account,pd.Timestamp("2025-01-06"),bar(10.5),market,config)
    check("sell_receipts_and_fees_reconcile",not account.positions and np.isclose(account.cash_balance,10000-1000-cash_charge(account.fills[0])+1050-cash_charge(account.fills[-1])))
    partial=CashStockAccount("ols",10000,portfolio,library);partial.start_day(day)
    partial.submit(order("partial","buy",200))
    execute_close_orders(partial,day,bar(volume=10000),market,config)
    check("volume_cap_partial_fill",partial.fills[0]["quantity"]==100 and partial.orders[0]["status"]=="partially_filled_then_expired")
    check("partial_remainder_reservation_released",partial.reserved_cash==0 and not partial.pending)
    limited=CashStockAccount("ols",1500,portfolio,library);limited.start_day(day)
    limited.submit(order("cash_limited","buy",200))
    execute_close_orders(limited,day,bar(10.5),market,config)
    check("actual_price_cash_recheck",limited.fills[0]["quantity"]==100 and limited.cash_available>=0)
    blocked=CashStockAccount("ols",10000,portfolio,library);blocked.start_day(day)
    blocked.submit(order("up","buy",100))
    execute_close_orders(blocked,day,bar(11.0),market,config)
    check("limit_up_buy_blocked",not blocked.fills and blocked.rejections[-1]["reason"]=="limit_up_buy_blocked")
    blocked.submit(order("halt","buy",100))
    execute_close_orders(blocked,day,bar(volume=0),market,config)
    check("zero_volume_blocked",not blocked.fills and blocked.rejections[-1]["reason"]=="no_observed_liquidity")
    blocked.submit(order("conditional","buy",100,conditional=True))
    execute_close_orders(blocked,day,bar(),market,config)
    check("conditional_buy_requires_real_exit",not blocked.fills)
    fees=copy.deepcopy(config)
    old=transaction_costs("sell",1000,10,"2023-08-25",fees)
    new=transaction_costs("sell",1000,10,"2023-08-28",fees)
    check("stamp_tax_date_change",old["stamp_tax"]==10 and new["stamp_tax"]==5)
    check("transfer_fee_date_change",transaction_costs("buy",1000,10,"2022-04-28",fees)["transfer_fee"]==.2
          and transaction_costs("buy",1000,10,"2022-04-29",fees)["transfer_fee"]==.1)
    check("buyer_has_no_stamp_tax",transaction_costs("buy",1000,10,day,fees)["stamp_tax"]==0)
    check("commission_minimum",transaction_costs("buy",100,10,day,fees)["commission"]==5)
    zero=dict(config,zero_cost_test=True)
    rights=CashStockAccount("ols",10000,portfolio,library);rights.start_day("2025-01-02")
    buy=order("rights_buy","buy",100,execution="2025-01-02")
    rights.submit(buy);rights.apply_fill(rights.pending[buy["order_id"]],"2025-01-02",100,10,transaction_costs("buy",100,10,"2025-01-02",zero))
    events=pd.DataFrame([dict(event_id="D1",event_type="dividend_record",event_date=pd.Timestamp("2025-01-02"),
        instrument=code,ex_date="2025-01-03",pay_date="2025-01-06",net_cash_per_share=.1)])
    process_company_events(rights,pd.Timestamp("2025-01-02"),events,"record_close")
    check("record_date_rights_do_not_prepay",rights.receivable_value==0 and rights.cash_balance==9000)
    rights.start_day(day);process_company_events(rights,day,events,"open");rights.mark_close(day,bar(9.9))
    check("ex_date_receivable_preserves_value",np.isclose(rights.nav,10000) and rights.receivable_value==10)
    sell=order("rights_sell","sell",100);rights.submit(sell)
    rights.apply_fill(rights.pending[sell["order_id"]],day,100,9.9,transaction_costs("sell",100,9.9,day,zero))
    process_company_events(rights,pd.Timestamp("2025-01-06"),events,"open")
    check("dividend_paid_after_position_sold",not rights.positions and rights.receivable_value==0 and np.isclose(rights.cash_balance,10000))
    split=CashStockAccount("ols",10000,portfolio,library);split.start_day("2025-01-02")
    split.submit(order("split_buy","buy",100,execution="2025-01-02"))
    split.apply_fill(split.pending["split_buy"],"2025-01-02",100,10,transaction_costs("buy",100,10,"2025-01-02",zero))
    split.start_day(day)
    split_events=pd.DataFrame([dict(event_id="S1",event_type="split",event_date=day,instrument=code,quantity_multiplier=2.0)])
    process_company_events(split,day,split_events,"open")
    check("split_quantity_price_and_nav",split.positions[code]["quantity"]==200 and split.positions[code]["valuation_price"]==5 and np.isclose(split.nav,10000))
    rejects("duplicate_company_event_rejected",lambda:process_company_events(split,day,split_events,"open"))
    stale_nav=split.nav;split.mark_close(day,pd.DataFrame(columns=bar().columns))
    check("missing_quote_keeps_position",split.positions[code]["quantity"]==200 and split.nav==stale_nav and split.daily_records[-1]["stale_position_count"]==1)
    # 一个完全可手算的事件循环：恒定价格、零费用，换仓不能产生收益。
    dates=pd.DatetimeIndex(["2025-01-02","2025-01-03","2025-01-06","2025-01-07","2025-01-08"])
    codes=["000001.SZ","000002.SZ","000004.SZ","000005.SZ"]
    frame_rows=[];signal_rows=[]
    fixture_cfg=copy.deepcopy(zero);fixture_cfg.update(start="2025-01-02",end="2025-01-08",volume_participation=1.0,progress_every=0)
    fixture_cfg["portfolio"].update(top_k=2,max_discretionary_replacements=1,single_name_weight_cap=1.0,min_cross_section=2)
    for i,date in enumerate(dates):
        for j,instrument in enumerate(codes):
            frame_rows.append(dict(signal_date=date,instrument=instrument,close=10.0,pre_close=10.0,
                volume_shares=1e6,amount_yuan=1e7,adv20=1e7,sigma20=.02,market_observed=True,
                historical_status_verified=True,known_can_buy=True,known_can_sell=True,upper_limit=11.0,lower_limit=9.0,
                adjustment_changed=False,adj_factor=1.0,previous_adj_factor=1.0))
            score=(4-j) if i==0 else (j+1)
            signal_rows.append(dict(signal_date=date,instrument=instrument,model="ols",score_raw=float(score),
                score_rank=1.0,candidate_eligible=True,cross_section_informative=True,
                earliest_execution_date=dates[i+1] if i+1<len(dates) else pd.NaT,
                signal_available_at=library["as_shanghai"](date+pd.Timedelta(hours=15,minutes=30)),
                result_level="test_fixture",source_hash="fixture"))
    signals=pd.DataFrame(signal_rows)
    fake_market=dict(daily=pd.DataFrame(frame_rows).set_index(["signal_date","instrument"]),calendar=dates,
                     actions=pd.DataFrame(columns=["event_id","event_type","event_date","instrument"]),rules=None)
    result=run_one_model("ols",{},fake_market,library,fixture_cfg,signal_frames=signals)
    check("constant_prices_zero_cost_no_profit",np.allclose(result["daily"].nav,fixture_cfg["initial_cash"]))
    check("signal_day_precedes_every_fill",(pd.to_datetime(result["fills"].signal_date)<pd.to_datetime(result["fills"].execution_date)).all())
    check("end_positions_not_forced_liquidated",result["terminal_snapshot"]["positions"].quantity.sum()>0)
    changed=signals.copy();changed.loc[changed.signal_date.ge(dates[3]),"score_raw"]*=-1
    changed_result=run_one_model("ols",{},fake_market,library,fixture_cfg,signal_frames=changed)
    earlier=lambda result:result["orders"].loc[pd.to_datetime(result["orders"].signal_date).lt(dates[3]),["order_id","instrument","side","requested_quantity"]].reset_index(drop=True)
    check("future_scores_do_not_change_past_orders",earlier(result).equals(earlier(changed_result)))
    bad_signals=signals.assign(label=999.0)
    rejects("future_labels_rejected",lambda:run_one_model("ols",{},fake_market,library,fixture_cfg,signal_frames=bad_signals))
    rejected_config=dict(fixture_cfg,mode="strict")
    rejects("strict_missing_inputs_rejected",lambda:validate_backtest_config(rejected_config,
        {"models":["ols"],"metadata":{"result_level":"research_only"}},fake_market))
    return pd.DataFrame(rows)


In [ ]:
if RUN_ACCEPTANCE_TESTS:
    acceptance_report = run_backtest_acceptance(signal_lib)
    display(acceptance_report)
    print(f"全部 {len(acceptance_report)} 项账本与时间边界测试通过。")


## 12. 运行固定 OLS 首季度试运行

默认从2025-01-02到2025-03-31，每个交易日正常推进。选择 OLS 只是固定的接口基线；没有根据测试收益选取模型。

这次运行会写入 `data/processed/backtest/pilot/run_.../`，记录真实信号触发的所有交易。它使用研究价格收益口径，任何未建模公司行动和估值陈旧都单独报告。费用和交易限制会影响实际资金与持仓。

重新运行会生成新目录，保留旧结果；查看旧运行可以直接调用 `load_backtest_run`，无需重训或重做信号。

In [ ]:
if RUN_PILOT:
    PILOT_RUN, pilot_summary, pilot_curves = run_backtest(
        bt_source, bt_market, signal_lib, BT_CONFIG, OUTPUT_ROOT / "pilot"
    )
    print("试运行目录：", PILOT_RUN)
    display(pilot_summary[["model", "trading_days", "total_return", "max_drawdown", "orders", "fills",
                           "total_trading_cost", "unmodeled_adjustment_events", "result_level"]])
else:
    print("试运行关闭；需要时把 RUN_PILOT 改为 True。")


## 13. 重新加载、核对账本并查看结果

本格重新检查保存文件的哈希，展示订单和实际成交；目标与实际持仓分开查看。金额单位元、收益小数0.01表示1%。

第一项核对是现金加持仓加应收是否等于NAV，第二项是可用现金与预留现金是否等于现金总额。然后查看未成交原因、持仓复权事件和仓位超限，避免只看净值曲线。

图表标题明确标记研究口径。图中价格收益账户与沪深300价格指数对照，不宣称已经完成股息再投资总收益比较。

In [ ]:
if RUN_PILOT:
    loaded_pilot = load_backtest_run(PILOT_RUN, signal_lib)
    PILOT_VIEW_MODEL = BT_CONFIG["models"][0]
    account_table = pd.read_parquet(PILOT_RUN / PILOT_VIEW_MODEL / "account.parquet")
    assert np.allclose(account_table.nav,
        account_table.cash_balance + account_table.market_value + account_table.receivables)
    assert np.allclose(account_table.cash_balance,
        account_table.cash_available + account_table.cash_reserved)
    orders_table = pd.read_parquet(PILOT_RUN / PILOT_VIEW_MODEL / "orders.parquet")
    fills_table = pd.read_parquet(PILOT_RUN / PILOT_VIEW_MODEL / "fills.parquet")
    display(account_table.head(5))
    display(orders_table[["order_id", "signal_date", "execution_date", "instrument", "side",
                          "requested_quantity", "filled_quantity", "status"]].head(10))
    display(fills_table[["execution_date", "instrument", "side", "quantity", "reference_close",
                         "commission", "stamp_tax", "transfer_fee", "execution_loss"]].head(10))
    audit = pd.read_parquet(PILOT_RUN / PILOT_VIEW_MODEL / "unmodeled_company_actions.parquet")
    display(audit.head(10))
    rejection_table = pd.read_parquet(PILOT_RUN / PILOT_VIEW_MODEL / "rejections.parquet")
    if "reason" in rejection_table:
        display(rejection_table.groupby("reason").size().rename("events").to_frame())
    print("通过：重新读取的日账户恒等式成立。")

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    loaded_pilot["curves"].plot(ax=axes[0, 0], title="Research price-account NAV")
    account_table.drawdown.plot(ax=axes[0, 1], title="Drawdown")
    account_table[["cash_balance", "market_value", "receivables"]].plot(ax=axes[1, 0], title="Account composition")
    account_table[["gross_turnover", "one_way_turnover"]].plot(ax=axes[1, 1], title="Realized turnover")
    plt.tight_layout()
    figure_dir = PILOT_RUN / "figures"
    figure_dir.mkdir(exist_ok=True)
    fig.savefig(figure_dir / "account_overview.png", dpi=150)
    plt.show()


## 14. 按需开启完整六模型同引擎比较

下面只切换模型名单与日期，组合、费用、资金、规则和账本完全相同。六模型各有独立账户，逐个运行以控制内存；不会重训模型或根据期末收益修改参数。

完整信号期目前为2025年。`RUN_FULL_BENCHMARK=True` 才执行，结果保存在单独的 benchmark 目录，不覆盖试运行。它仍继承相同的研究口径与数据限制。

后续增加成本敏感性、同池等权策略或全收益基准时，应登记为独立版本，并展示全部情景，不只保留最好曲线。

In [ ]:
if RUN_FULL_BENCHMARK:
    FULL_CONFIG = copy.deepcopy(BT_CONFIG)
    FULL_CONFIG["models"] = bt_source["models"].copy()
    FULL_CONFIG["start"] = str(bt_source["report"].signal_date.min().date())
    FULL_CONFIG["end"] = str(bt_source["report"].signal_date.max().date())
    full_market = load_backtest_market(bt_source, signal_lib, FULL_CONFIG["start"], FULL_CONFIG["end"],
        HISTORICAL_STATUS_PATH, CORPORATE_ACTIONS_PATH, HISTORICAL_QUANTITY_RULES_PATH)
    BENCHMARK_RUN, benchmark_summary, benchmark_curves = run_backtest(
        bt_source, full_market, signal_lib, FULL_CONFIG, OUTPUT_ROOT / "benchmark")
    display(benchmark_summary)
    print("六模型比较目录：", BENCHMARK_RUN)
else:
    print("完整六模型回测未执行；框架接口已提供，按需设置 RUN_FULL_BENCHMARK=True。")


## 15. 输出文件与下一步

每个运行保存：

```text
config.json / source.json / metadata.json
performance_summary.csv / nav_comparison.parquet / benchmark.parquet
<model>/
    account.parquet / daily_returns.parquet
    orders.parquet / fills.parquet / rejections.parquet
    positions.parquet / target_positions.parquet / decisions.parquet
    company_events.parquet / unmodeled_company_actions.parquet
    constraint_breaches.parquet / terminal_snapshot.json
    pending_dividend_rights.json
```

`metadata.status=complete` 表示这次计算与保存完成，不表示历史交易状态、净分红和执行精度已经得到独立实盘验证。期末账户保留真实持仓，没有暗中按最后收盘价全部变现。

下一步优先补历史状态、每日真实涨跌停价与公司行动数据，再核对数量规则、净分红与税费，逐步推进到严格输入版。当前框架的接口、订单、成交和账户已经连通，可以在同一引擎下继续加入新的组合规则与基准。

参考：[财政部2023年印花税减半公告](https://m.mof.gov.cn/czxw/202308/t20230827_3904226.htm)、[上交所科创板历史规则投教](https://edu.sse.com.cn/tib/qa/c/4866268.shtml)、[中国结算历史事项](https://m.chinaclear.cn/zdjs/dsj/about_dsj.shtml)。实际接入历史费率和特殊规则时应归档相应有效版本，不能直接把2026年规则回填2025年。

整体设计和数据缺口见项目的 `bactest_benchmark.md`，信号接口约定见 `signal.md`。